## Scraper — Estatisticas por Jogo (Basketball Reference)

Fonte: https://www.basketball-reference.com/leagues/NBA_2025_per_game.html
Saida: basket_dbt/seeds/players_stats.csv

In [1]:
from __future__ import annotations
import logging, time
from io import StringIO
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup, Comment
from selenium import webdriver
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.chrome.service import Service

In [2]:
logger = logging.getLogger('bbr_stats')
if not logger.handlers:
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', '%Y-%m-%d %H:%M:%S'))
    logger.addHandler(h)
logger.setLevel(logging.INFO)

In [3]:
CHROMEDRIVER_PATH = '/snap/bin/chromium.chromedriver'
CHROME_BINARY     = '/usr/bin/chromium-browser'

# Renomeia colunas com caracteres invalidos em SQL (%, inicio com numero)
COLUMN_RENAME = {
    'FG%':  'fg_pct',
    '3P':   'three_p',
    '3PA':  'three_pa',
    '3P%':  'three_p_pct',
    '2P':   'two_p',
    '2PA':  'two_pa',
    '2P%':  'two_p_pct',
    'eFG%': 'efg_pct',
    'FT%':  'ft_pct',
}

def get_rendered_html(url: str, wait_seconds: int = 8) -> str:
    logger.info('Abrindo: %s', url)
    opts = ChromeOptions()
    opts.binary_location = CHROME_BINARY
    opts.add_argument('--headless=new')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--window-size=1920,1080')
    opts.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/127.0.0.0 Safari/537.36')
    driver = webdriver.Chrome(service=Service(CHROMEDRIVER_PATH), options=opts)
    try:
        driver.get(url)
        time.sleep(wait_seconds)
        html = driver.page_source
        logger.info('HTML obtido (%d chars).', len(html))
        return html
    finally:
        driver.quit()

def uncomment_tables(raw_html: str) -> BeautifulSoup:
    soup = BeautifulSoup(raw_html, 'lxml')
    comments = soup.find_all(string=lambda t: isinstance(t, Comment))
    for c in comments:
        c.replace_with(BeautifulSoup(c, 'lxml'))
    logger.info('%d comentario(s) descomentados.', len(comments))
    return soup

def extract_table(soup: BeautifulSoup, table_id: str) -> pd.DataFrame:
    table = soup.select_one(f'table#{table_id}')
    if not table:
        raise RuntimeError(f'Tabela {table_id!r} nao encontrada.')
    df = pd.read_html(StringIO(str(table)))[0]
    logger.info('Tabela %r: %d linhas, %d colunas.', table_id, *df.shape)
    return df

In [4]:
SEASON   = 2025
URL      = f'https://www.basketball-reference.com/leagues/NBA_{SEASON}_per_game.html'
TABLE_ID = 'per_game_stats'
OUT_PATH = Path('../../basket_dbt/seeds/players_stats.csv')

html   = get_rendered_html(url=URL)
soup   = uncomment_tables(html)
df_raw = extract_table(soup, TABLE_ID)

2026-04-03 14:37:07 | INFO | Abrindo: https://www.basketball-reference.com/leagues/NBA_2025_per_game.html


2026-04-03 14:37:18 | INFO | HTML obtido (2538131 chars).


2026-04-03 14:37:19 | INFO | 136 comentario(s) descomentados.


2026-04-03 14:37:19 | INFO | Tabela 'per_game_stats': 756 linhas, 31 colunas.


In [5]:
# Remove cabecalhos repetidos, League Average, Rk e Awards
df = df_raw[
    (df_raw['Player'] != 'Player') &
    (df_raw['Player'] != 'League Average')
].drop(columns=['Rk', 'Awards'], errors='ignore').copy().reset_index(drop=True)

# Renomeia colunas invalidas para SQL
df = df.rename(columns=COLUMN_RENAME)

logger.info('Linhas: %d — Colunas: %s', len(df), df.columns.tolist())
print(df.head(3).to_string())

2026-04-03 14:37:19 | INFO | Linhas: 735 — Colunas: ['Player', 'Age', 'Team', 'Pos', 'G', 'GS', 'MP', 'FG', 'FGA', 'fg_pct', 'three_p', 'three_pa', 'three_p_pct', 'two_p', 'two_pa', 'two_p_pct', 'efg_pct', 'FT', 'FTA', 'ft_pct', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']


                    Player Age Team Pos   G  GS    MP    FG   FGA fg_pct three_p three_pa three_p_pct two_p two_pa two_p_pct efg_pct   FT   FTA ft_pct  ORB  DRB   TRB   AST  STL  BLK  TOV   PF   PTS
0  Shai Gilgeous-Alexander  26  OKC  PG  76  76  34.2  11.3  21.8   .519     2.1      5.7        .375   9.2   16.1      .571    .569  7.9   8.8   .898  0.9  4.1   5.0   6.4  1.7  1.0  2.4  2.2  32.7
1    Giannis Antetokounmpo  30  MIL  PF  67  67  34.2  11.8  19.7   .601     0.2      0.9        .222  11.6   18.7      .620    .607  6.5  10.6   .617  2.2  9.7  11.9   6.5  0.9  1.2  3.1  2.3  30.4
2             Nikola Jokić  29  DEN   C  70  70  36.7  11.2  19.5   .576     2.0      4.7        .417   9.3   14.8      .627    .627  5.2   6.4   .800  2.9  9.9  12.7  10.2  1.8  0.6  3.3  2.3  29.6


In [6]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False, encoding='utf-8')
logger.info('Salvo em: %s (%d linhas)', OUT_PATH.resolve(), len(df))

2026-04-03 14:37:19 | INFO | Salvo em: /home/henri/Basketanalysis/basket_dbt/seeds/players_stats.csv (735 linhas)
